In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("ClusteringModel").getOrCreate()


In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df)


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=iris.feature_names,
    outputCol="features"
)
final_df = assembler.transform(spark_df).select("features")


In [ ]:
from pyspark.ml.clustering import KMeans

# Define KMeans with 3 clusters (since Iris has 3 species)
kmeans = KMeans(featuresCol="features", k=3, seed=42)
model = kmeans.fit(final_df)

# Make predictions
predictions = model.transform(final_df)


In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction")
silhouette = evaluator.evaluate(predictions)
print("Silhouette Score:", silhouette)


Silhouette Score: 0.7344130579787836


In [ ]:
centers = model.clusterCenters()
print("Cluster Centers:")
for center in centers:
    print(center)


Cluster Centers:
[5.88360656 2.74098361 4.38852459 1.43442623]
[5.006 3.428 1.462 0.246]
[6.85384615 3.07692308 5.71538462 2.05384615]
